In [1]:
# ===== 1. Imports =====
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from math import sqrt

from xgboost import XGBRegressor

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import LSTM, Dense, Input, RepeatVector, TimeDistributed
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler

# ===== 2. Recreate Milestone 3 + 4 Data (same as previous notebooks) =====
players = ['Alice', 'Bob', 'Charlie', 'David', 'Alice']
ages = [24, 28, 26, 30, 22]
goals = [10, 15, 8, 9, 7]
contractyears = [2, 1, 3, 2, 2]
goalspast3seasons = [7, 13, 5, 7, 9]
injuries = [1, 0, 3, 1, 2]
gamesplayed = [25, 34, 22, 30, 25]

milestonedf = pd.DataFrame({
    'player': players,
    'age': ages,
    'goals': goals,
    'contractyears': contractyears,
    'goalspast3seasons': goalspast3seasons,
    'injuries': injuries,
    'gamesplayed': gamesplayed
})

milestonedf['goaltrend'] = (milestonedf['goals'] - milestonedf['goalspast3seasons']) / milestonedf['goalspast3seasons']
milestonedf['injuryrisk'] = milestonedf['injuries'] / milestonedf['gamesplayed']
milestonedf['contractleft'] = milestonedf['contractyears'] - 1

# Sentiment features (from Milestone 3)
tweets = [
    "Alice's performance was excellent today!",
    "Bob was off his game, seemed distracted.",
    "Charlie back from injury--solid match.",
    "David's contract negotiations affecting focus.",
    "Alice continues to impress season after season."
]
sentplayers = ['Alice', 'Bob', 'Charlie', 'David', 'Alice']

from textblob import TextBlob
sentdf = pd.DataFrame({'player': sentplayers, 'tweet': tweets})
sentdf['polarity'] = sentdf['tweet'].apply(lambda x: TextBlob(x).sentiment.polarity)
playersentiment = sentdf.groupby('player')['polarity'].mean().reset_index().rename(columns={'polarity':'sentimentscore'})
milestonedf = pd.merge(milestonedf, playersentiment, on='player', how='left')
milestonedf['expectedmarketvalueboost'] = milestonedf['sentimentscore'] * 1.5

# Transfer value as in Milestone 4
milestonedf['transfervalue'] = milestonedf['goals'] * 1e6 + milestonedf['expectedmarketvalueboost'] * 1e6

print("Final Milestone Data:")
print(milestonedf)


Final Milestone Data:
    player  age  goals  contractyears  goalspast3seasons  injuries  \
0    Alice   24     10              2                  7         1   
1      Bob   28     15              1                 13         0   
2  Charlie   26      8              3                  5         3   
3    David   30      9              2                  7         1   
4    Alice   22      7              2                  9         2   

   gamesplayed  goaltrend  injuryrisk  contractleft  sentimentscore  \
0           25   0.428571    0.040000             1             0.5   
1           34   0.153846    0.000000             0            -0.4   
2           22   0.600000    0.136364             2             0.0   
3           30   0.285714    0.033333             1             0.0   
4           25  -0.222222    0.080000             1             0.5   

   expectedmarketvalueboost  transfervalue  
0                      0.75     10750000.0  
1                     -0.60     14400000

In [2]:
# ===== LSTM DATA PREP (Milestone 4 style) =====
seqlen = 2

# Univariate LSTM: goals -> transfervalue
scaler_uni = MinMaxScaler()
uni_features = milestonedf['goals'].values.reshape(-1, 1)
uni_target = milestonedf['transfervalue'].values.reshape(-1, 1)

uni_features_scaled = scaler_uni.fit_transform(uni_features)
uni_target_scaled = scaler_uni.fit_transform(uni_target)

X_uni, y_uni = [], []
for i in range(len(uni_features_scaled) - seqlen):
    X_uni.append(uni_features_scaled[i:i+seqlen])
    y_uni.append(uni_target_scaled[i+seqlen, 0])
X_uni = np.array(X_uni)
y_uni = np.array(y_uni)

# Multivariate LSTM: [goals, injuries, sentimentscore] -> transfervalue
multi_features = milestonedf[['goals', 'injuries', 'sentimentscore']].values
scaler_multi = MinMaxScaler()
multi_features_scaled = scaler_multi.fit_transform(multi_features)

X_multi, y_multi = [], []
for i in range(len(multi_features_scaled) - seqlen):
    X_multi.append(multi_features_scaled[i:i+seqlen])
    y_multi.append(uni_target_scaled[i+seqlen, 0])
X_multi = np.array(X_multi)
y_multi = np.array(y_multi)

# Encoder-Decoder LSTM: multi-step (2-step)
TIMESTEPS = seqlen
FUTURESTEPS = 2
X_encdec, y_encdec = [], []
features_encdec = multi_features_scaled
label_encdec = uni_target_scaled

for i in range(len(features_encdec) - TIMESTEPS - FUTURESTEPS + 1):
    X_encdec.append(features_encdec[i:i+TIMESTEPS])
    y_encdec.append(label_encdec[i+TIMESTEPS:i+TIMESTEPS+FUTURESTEPS, 0])

X_encdec = np.array(X_encdec)
y_encdec = np.array(y_encdec)

# Baseline LSTMs (single config)
def build_univariate_lstm(units=32, lr=0.001):
    model = Sequential([
        LSTM(units, activation='relu', input_shape=(seqlen, 1)),
        Dense(1)
    ])
    model.compile(optimizer=Adam(lr), loss='mse', metrics=['mae'])
    return model

def build_multivariate_lstm(units=32, lr=0.001):
    model = Sequential([
        LSTM(units, activation='relu', input_shape=(seqlen, 3)),
        Dense(1)
    ])
    model.compile(optimizer=Adam(lr), loss='mse', metrics=['mae'])
    return model

def build_encdec_lstm(units=32, lr=0.001):
    inputs = Input(shape=(TIMESTEPS, 3))
    encoded = LSTM(units, activation='relu')(inputs)
    decoded = RepeatVector(FUTURESTEPS)(encoded)
    decoded = LSTM(units, activation='relu', return_sequences=True)(decoded)
    outputs = TimeDistributed(Dense(1))(decoded)
    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(lr), loss='mse')
    return model

model_uni = build_univariate_lstm()
model_multi = build_multivariate_lstm()
model_encdec = build_encdec_lstm()

history_uni = model_uni.fit(X_uni, y_uni, epochs=30, verbose=0)
history_multi = model_multi.fit(X_multi, y_multi, epochs=30, verbose=0)
history_encdec = model_encdec.fit(X_encdec, y_encdec, epochs=30, verbose=0)

uni_pred_scaled = model_uni.predict(X_uni)
multi_pred_scaled = model_multi.predict(X_multi)
encdec_pred_scaled = model_encdec.predict(X_encdec)

# Inverse scale to original transfervalue
uni_pred = scaler_uni.inverse_transform(uni_pred_scaled)
multi_pred = scaler_uni.inverse_transform(multi_pred_scaled)
encdec_pred = scaler_uni.inverse_transform(encdec_pred_scaled.reshape(-1, 1)).reshape(encdec_pred_scaled.shape[0], FUTURESTEPS)
true_uni = scaler_uni.inverse_transform(y_uni.reshape(-1, 1))
true_multi = scaler_uni.inverse_transform(y_multi.reshape(-1, 1))
true_encdec = scaler_uni.inverse_transform(y_encdec.reshape(-1, FUTURESTEPS))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 177ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 333ms/step


In [3]:
def print_regression_metrics(name, y_true, y_pred):
    rmse = sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{name} -> RMSE: {rmse:.4f}, MAE: {mae:.4f}, R2: {r2:.4f}")

print("=== Baseline LSTM Metrics ===")
print_regression_metrics("Univariate LSTM", true_uni, uni_pred)
print_regression_metrics("Multivariate LSTM", true_multi, multi_pred)
print_regression_metrics("Encoder-Decoder LSTM (avg over 2 steps)",
                         true_encdec.mean(axis=1),
                         encdec_pred.mean(axis=1))


=== Baseline LSTM Metrics ===
Univariate LSTM -> RMSE: 545091.1275, MAE: 468054.5000, R2: -0.0187
Multivariate LSTM -> RMSE: 452889.5163, MAE: 437827.3333, R2: 0.2968
Encoder-Decoder LSTM (avg over 2 steps) -> RMSE: 59231.9014, MAE: 45018.7500, R2: 0.1018


In [8]:
from sklearn.model_selection import train_test_split
from math import sqrt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
import numpy as np

def print_regression_metrics(name, y_true, y_pred):
    rmse = sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{name} -> RMSE: {rmse:.4f}, MAE: {mae:.4f}, R2: {r2:.4f}")

features = ['goals', 'injuries', 'sentimentscore', 'expectedmarketvalueboost',
            'gamesplayed', 'goaltrend', 'injuryrisk']
target = 'transfervalue'

# rows that correspond to LSTM output (start at seqlen)
effective_idx = list(range(seqlen, len(milestonedf)))  # e.g. [2,3,4]
X_full = milestonedf.iloc[effective_idx][features].values
y_full = milestonedf.iloc[effective_idx][target].values

# LSTM preds
uni_eff    = uni_pred.ravel()
multi_eff  = multi_pred.ravel()
encdec_eff = encdec_pred.mean(axis=1)

# make all arrays the same length = min length
min_len = min(len(X_full), len(y_full), len(uni_eff), len(multi_eff), len(encdec_eff))
X_full     = X_full[:min_len]
y_full     = y_full[:min_len]
uni_eff    = uni_eff[:min_len]
multi_eff  = multi_eff[:min_len]
encdec_eff = encdec_eff[:min_len]

# now split – all arrays have identical length
X_train, X_val, y_train, y_val, \
uni_train, uni_val, \
multi_train, multi_val, \
encdec_train, encdec_val = train_test_split(
    X_full, y_full,
    uni_eff, multi_eff, encdec_eff,
    test_size=0.4,
    random_state=42
)

# baseline XGBoost
xgb_base = XGBRegressor(n_estimators=25, max_depth=3, seed=42)
xgb_base.fit(X_train, y_train)
y_pred_xgb = xgb_base.predict(X_val)
print("\n=== Baseline XGBoost Metrics (validation) ===")
print_regression_metrics("XGBoost", y_val, y_pred_xgb)

# stacked ensemble
X_stack_train = np.column_stack([X_train, uni_train, multi_train, encdec_train])
X_stack_val   = np.column_stack([X_val,   uni_val,   multi_val,   encdec_val])

xgb_ens_base = XGBRegressor(n_estimators=15, max_depth=3, seed=24)
xgb_ens_base.fit(X_stack_train, y_train)
y_pred_ens = xgb_ens_base.predict(X_stack_val)
print_regression_metrics("Ensemble (XGBoost + LSTM)", y_val, y_pred_ens)



=== Baseline XGBoost Metrics (validation) ===
XGBoost -> RMSE: 1000000.0000, MAE: 1000000.0000, R2: nan
Ensemble (XGBoost + LSTM) -> RMSE: 1000000.0000, MAE: 1000000.0000, R2: nan


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_regression.py:1266: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_regression.py:1266: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


In [9]:
lstm_configs = [
    {'units': 16, 'lr': 0.001, 'epochs': 20},
    {'units': 32, 'lr': 0.001, 'epochs': 30},
    {'units': 64, 'lr': 0.0005, 'epochs': 40},
]

best_uni = None
best_uni_score = 1e18

for cfg in lstm_configs:
    m = build_univariate_lstm(units=cfg['units'], lr=cfg['lr'])
    h = m.fit(X_uni, y_uni, epochs=cfg['epochs'], verbose=0)
    preds = scaler_uni.inverse_transform(m.predict(X_uni))
    score = sqrt(mean_squared_error(true_uni, preds))
    print(f"UNI cfg {cfg} -> RMSE {score:.4f}")
    if score < best_uni_score:
        best_uni_score = score
        best_uni = (cfg, m)

best_multi = None
best_multi_score = 1e18
for cfg in lstm_configs:
    m = build_multivariate_lstm(units=cfg['units'], lr=cfg['lr'])
    h = m.fit(X_multi, y_multi, epochs=cfg['epochs'], verbose=0)
    preds = scaler_uni.inverse_transform(m.predict(X_multi))
    score = sqrt(mean_squared_error(true_multi, preds))
    print(f"MULTI cfg {cfg} -> RMSE {score:.4f}")
    if score < best_multi_score:
        best_multi_score = score
        best_multi = (cfg, m)

best_enc = None
best_enc_score = 1e18
for cfg in lstm_configs:
    m = build_encdec_lstm(units=cfg['units'], lr=cfg['lr'])
    h = m.fit(X_encdec, y_encdec, epochs=cfg['epochs'], verbose=0)
    preds_scaled = m.predict(X_encdec)
    preds = scaler_uni.inverse_transform(preds_scaled.reshape(-1, 1)).reshape(preds_scaled.shape[0], FUTURESTEPS)
    score = sqrt(mean_squared_error(true_encdec.mean(axis=1), preds.mean(axis=1)))
    print(f"ENCDEC cfg {cfg} -> RMSE {score:.4f}")
    if score < best_enc_score:
        best_enc_score = score
        best_enc = (cfg, m)

print("\nBest Uni LSTM config:", best_uni[0])
print("Best Multi LSTM config:", best_multi[0])
print("Best Enc-Dec LSTM config:", best_enc[0])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 456ms/step
UNI cfg {'units': 16, 'lr': 0.001, 'epochs': 20} -> RMSE 586576.7694


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 172ms/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


...

UNI cfg {'units': 64, 'lr': 0.0005, 'epochs': 40} -> RMSE 493172.5369
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 246ms/step
MULTI cfg {'units': 16, 'lr': 0.001, 'epochs': 20} -> RMSE 732182.2775


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 170ms/step
MULTI cfg {'units': 32, 'lr': 0.001, 'epochs': 30} -> RMSE 484603.2822


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step
MULTI cfg {'units': 64, 'lr': 0.0005, 'epochs': 40} -> RMSE 491773.9241
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 368ms/step
ENCDEC cfg {'units': 16, 'lr': 0.001, 'epochs': 20} -> RMSE 273007.9656
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 329ms/step
ENCDEC cfg {'units': 32, 'lr': 0.001, 'epochs': 30} -> RMSE 96064.2138
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 336ms/step
ENCDEC cfg {'units': 64, 'lr': 0.0005, 'epochs': 40} -> RMSE 138714.9532

Best Uni LSTM config: {'units': 64, 'lr': 0.0005, 'epochs': 40}
Best Multi LSTM config: {'units': 32, 'lr': 0.001, 'epochs': 30}
Best Enc-Dec LSTM config: {'units': 32, 'lr': 0.001, 'epochs': 30}


In [10]:
xgb_params = [
    {'n_estimators': 20, 'max_depth': 2},
    {'n_estimators': 50, 'max_depth': 3},
    {'n_estimators': 100, 'max_depth': 4},
]

best_xgb = None
best_xgb_rmse = 1e18

for params in xgb_params:
    model = XGBRegressor(**params, seed=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    rmse = sqrt(mean_squared_error(y_val, preds))
    print(f"XGB cfg {params} -> RMSE {rmse:.4f}")
    if rmse < best_xgb_rmse:
        best_xgb_rmse = rmse
        best_xgb = (params, model)

best_ens = None
best_ens_rmse = 1e18
for params in xgb_params:
    model = XGBRegressor(**params, seed=24)
    model.fit(X_stack_train, y_train)
    preds = model.predict(X_stack_val)
    rmse = sqrt(mean_squared_error(y_val, preds))
    print(f"Ensemble cfg {params} -> RMSE {rmse:.4f}")
    if rmse < best_ens_rmse:
        best_ens_rmse = rmse
        best_ens = (params, model)

print("\nBest XGBoost config:", best_xgb[0])
print("Best Ensemble config:", best_ens[0])


XGB cfg {'n_estimators': 20, 'max_depth': 2} -> RMSE 1000000.0000
XGB cfg {'n_estimators': 50, 'max_depth': 3} -> RMSE 1000000.0000
XGB cfg {'n_estimators': 100, 'max_depth': 4} -> RMSE 1000000.0000
Ensemble cfg {'n_estimators': 20, 'max_depth': 2} -> RMSE 1000000.0000
Ensemble cfg {'n_estimators': 50, 'max_depth': 3} -> RMSE 1000000.0000
Ensemble cfg {'n_estimators': 100, 'max_depth': 4} -> RMSE 1000000.0000

Best XGBoost config: {'n_estimators': 20, 'max_depth': 2}
Best Ensemble config: {'n_estimators': 20, 'max_depth': 2}


In [11]:
# Use best tuned models for final evaluation
best_uni_cfg, best_uni_model = best_uni
best_multi_cfg, best_multi_model = best_multi
best_enc_cfg, best_enc_model = best_enc
best_xgb_cfg, best_xgb_model = best_xgb
best_ens_cfg, best_ens_model = best_ens

# Final predictions
final_uni_pred = scaler_uni.inverse_transform(best_uni_model.predict(X_uni))
final_multi_pred = scaler_uni.inverse_transform(best_multi_model.predict(X_multi))
final_enc_pred_scaled = best_enc_model.predict(X_encdec)
final_enc_pred = scaler_uni.inverse_transform(final_enc_pred_scaled.reshape(-1,1)).reshape(final_enc_pred_scaled.shape[0], FUTURESTEPS)

final_xgb_pred = best_xgb_model.predict(X_val)
final_ens_pred = best_ens_model.predict(X_stack_val)

print("=== Final Tuned LSTM Metrics ===")
print_regression_metrics(f"Tuned Univariate LSTM {best_uni_cfg}", true_uni, final_uni_pred)
print_regression_metrics(f"Tuned Multivariate LSTM {best_multi_cfg}", true_multi, final_multi_pred)
print_regression_metrics(f"Tuned Enc-Dec LSTM {best_enc_cfg}",
                         true_encdec.mean(axis=1),
                         final_enc_pred.mean(axis=1))

print("\n=== Final Tuned XGBoost / Ensemble (validation) ===")
print_regression_metrics(f"Tuned XGBoost {best_xgb_cfg}", y_val, final_xgb_pred)
print_regression_metrics(f"Tuned Ensemble {best_ens_cfg}", y_val, final_ens_pred)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
=== Final Tuned LSTM Metrics ===
Tuned Univariate LSTM {'units': 64, 'lr': 0.0005, 'epochs': 40} -> RMSE: 493172.5369, MAE: 465978.0000, R2: 0.1661
Tuned Multivariate LSTM {'units': 32, 'lr': 0.001, 'epochs': 30} -> RMSE: 484603.2822, MAE: 458819.6667, R2: 0.1948
Tuned Enc-Dec LSTM {'units': 32, 'lr': 0.001, 'epochs': 30} -> RMSE: 96064.2138, MAE: 76986.2500, R2: -1.3625

=== Final Tuned XGBoost / Ensemble (validation) ===
Tuned XGBoost {'n_estimators': 20, 'max_depth': 2} -> RMSE: 1000000.0000, MAE: 1000000.0000, R2: nan
Tuned Ensemble {'n_estimators': 20, 'max_depth': 2} -> RMSE: 1000000.0000, MAE: 1000000.0000, R2: nan


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_regression.py:1266: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_regression.py:1266: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
